In [48]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import warnings
import matplotlib.dates as mdates
from scipy.stats import norm as scipy_norm


warnings.filterwarnings('ignore')

In [49]:

PATHS = {
    'M0': '../output/results/M0_results_5YCDS.csv',
    'M1': '../output/results/M1_drift_results_5YCDS.csv',
    'M2': '../output/results/M2_results_5YCDS.csv',
}

# Adjust these to match your data
DATE_COL = 'date'
COUNTRY_COL = 'country'
DD_COL = 'distance_to_distress'
CDS_COL = 'cds_spread'

# Horizons in weeks for computing changes
HORIZONS = [1, 2, 4, 8]

# Define your groups — edit these lists to match your sample
EXPORTERS = [
    'Saudi Arabia',
    'UAE (Abu Dhabi)',
    'Qatar',
    'Colombia',
    'Mexico',
    'Brazil',
    'Egypt',
    'Malaysia']

CONTROLS = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa', 'South Korea', 'Thailand', 'Turkey']



RECOVERY = 0.40
HORIZON_CDS = 5

In [50]:
# ──────────────────────────────────────────────────────────────
# 2. LOAD DATA
# ──────────────────────────────────────────────────────────────

def load_models(paths):
    dfs = {}
    for m, p in paths.items():
        df = pd.read_csv(p, parse_dates=[DATE_COL])
        df = df.sort_values([COUNTRY_COL, DATE_COL])
        dfs[m] = df
    return dfs


def dd_to_implied_cds(dd_series, country, 
                       alpha_dict=ALPHA_CALIBRATION,
                       recovery=RECOVERY, horizon=HORIZON_CDS):
    alpha   = alpha_dict.get(country, np.nan)
    dd_cal  = dd_series * alpha
    pd_cal  = pd.Series(scipy_norm.cdf(-dd_cal.values),
                        index=dd_series.index).clip(1e-6, 1 - 1e-6)
    cds_imp = -np.log(1 - pd_cal) * (1 - recovery) / horizon * 10000
    return cds_imp

dfs = load_models(PATHS)

# Auto-detect controls if not specified
all_countries = sorted(dfs['M0'][COUNTRY_COL].unique())
if CONTROLS is None:
    CONTROLS = [c for c in all_countries if c not in EXPORTERS]

print(f"Total countries: {len(all_countries)}")
print(f"Exporters ({len(EXPORTERS)}): {EXPORTERS}")
print(f"Controls  ({len(CONTROLS)}): {CONTROLS}")

Total countries: 16
Exporters (8): ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
Controls  (8): ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa', 'South Korea', 'Thailand', 'Turkey']


In [46]:
# ── Replace compute_changes with this version ─────────────────
def compute_changes(df, horizon_weeks):
    """
    Compute log-difference of model-implied CDS and observed CDS
    over a given horizon (in weeks).
    Model-implied CDS is derived from DD via alpha calibration.
    """
    out = []
    for country, grp in df.groupby(COUNTRY_COL):
        g = grp.set_index(DATE_COL).sort_index()
        g = g[[DD_COL, CDS_COL]].resample('W').last().dropna()

        # Model-implied CDS from DD
        g['cds_implied'] = dd_to_implied_cds(g[DD_COL], country)

        # Log differences
        g['delta_dd']  = np.log(g['cds_implied']).diff(horizon_weeks)
        g['delta_cds'] = np.log(g[CDS_COL]).diff(horizon_weeks)

        g[COUNTRY_COL] = country
        out.append(g.dropna(subset=['delta_dd', 'delta_cds']).reset_index())

    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)

# ──────────────────────────────────────────────────────────────
# 4. CORRELATION FUNCTIONS
# ──────────────────────────────────────────────────────────────

def pearson_corr(dd, cds):
    if len(dd) < 10:
        return np.nan
    return np.corrcoef(dd, cds)[0, 1]


def spearman_corr(dd, cds):
    if len(dd) < 10:
        return np.nan
    rho, _ = spearmanr(dd, cds)
    return rho


def country_correlations(df, corr_func):
    """Compute correlation per country, return Series."""
    results = {}
    for country, grp in df.groupby(COUNTRY_COL):
        dd = grp['delta_dd'].values
        cds = grp['delta_cds'].values
        mask = np.isfinite(dd) & np.isfinite(cds)
        if mask.sum() >= 10:
            results[country] = corr_func(dd[mask], cds[mask])
    return pd.Series(results)


def run_correlation_table(corr_func, corr_name):
    """
    Returns a DataFrame:
        rows = horizons
        columns = MultiIndex (model, group)
        values = mean correlation across countries in that group
    """
    records = []
    for h in HORIZONS:
        for m in PATHS.keys():
            changes = compute_changes(dfs[m], h)
            if changes.empty:
                continue
            corrs = country_correlations(changes, corr_func)

            all_mean = corrs.mean()
            exp_mean = corrs[corrs.index.isin(EXPORTERS)].mean()
            ctrl_mean = corrs[corrs.index.isin(CONTROLS)].mean()

            records.append({
                'horizon_w': h,
                'model': m,
                'all': all_mean,
                'exporters': exp_mean,
                'controls': ctrl_mean,
                'diff': exp_mean - ctrl_mean,
            })

    df_out = pd.DataFrame(records)
    df_out['corr_type'] = corr_name
    return df_out

In [47]:
pearson_results = run_correlation_table(pearson_corr, 'Pearson')
# Spearman
spearman_results = run_correlation_table(spearman_corr, 'Spearman')

results = pd.concat([pearson_results, spearman_results], ignore_index=True)


KeyError: "['distance_to_distress'] not in index"

In [37]:
# 6. DISPLAY TABLES
# ──────────────────────────────────────────────────────────────

def display_table(df, corr_name, group_col, title):
    subset = df[df['corr_type'] == corr_name]
    pivot = subset.pivot_table(index='horizon_w', columns='model', values=group_col)
    pivot = pivot[list(PATHS.keys())]  # enforce column order
    pivot.index.name = 'Horizon (weeks)'
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"  {corr_name} correlation — average across countries")
    print(f"{'='*60}")
    print(pivot.round(3).to_string())
    return pivot


# Table 1: Pearson, all countries
t1 = display_table(results, 'Pearson', 'all',
                    'Pearson — All Countries')

# Table 2: Pearson, exporters vs controls
t2_exp = display_table(results, 'Pearson', 'exporters',
                        'Pearson — Exporters')
t2_ctrl = display_table(results, 'Pearson', 'controls',
                         'Pearson — Controls')

# Table 3: Spearman, exporters vs controls
t3_exp = display_table(results, 'Spearman', 'exporters',
                        'Spearman — Exporters')
t3_ctrl = display_table(results, 'Spearman', 'controls',
                         'Spearman — Controls')


  Pearson — All Countries
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1                0.031  0.126  0.301
2                0.052  0.193  0.363
4                0.081  0.210  0.419
8                0.122  0.267  0.469

  Pearson — Exporters
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1                0.037  0.198  0.278
2                0.092  0.269  0.336
4                0.136  0.263  0.416
8                0.168  0.281  0.457

  Pearson — Controls
  Pearson correlation — average across countries
model               M0     M1     M2
Horizon (weeks)                     
1                0.021  0.081  0.335
2                0.023  0.144  0.391
4                0.041  0.178  0.437
8                0.083  0.265  0.517

  Spearman — Exporters
  Spearman correlation — average across countries
model               M0     M1     M2
Hori

In [38]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.stats import rankdata
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────
# NEWEY-WEST CORRELATION TEST
# Regress delta_cds on delta_dd with HAC SE, lag = horizon - 1
# For Spearman: rank-transform both series first
# Returns: coef, t-stat, p-value, n
# ──────────────────────────────────────────────────────────────

def nw_corr_test(dd, cds, horizon, spearman=False):
    """
    Tests whether the correlation between dd and cds changes is
    significantly different from zero, accounting for overlap.
    
    NW lag = horizon - 1 (covers exactly the overlapping window).
    For Spearman: rank-transforms both series before regression.
    Returns (correlation, t_stat, p_value, n_obs)
    """
    mask = np.isfinite(dd) & np.isfinite(cds)
    dd, cds = dd[mask], cds[mask]
    n = len(dd)
    if n < 10:
        return np.nan, np.nan, np.nan, n

    if spearman:
        x = rankdata(dd).astype(float)
        y = rankdata(cds).astype(float)
        # Normalize ranks to [-1, 1] so slope ≈ Spearman rho
        x = (x - x.mean()) / x.std(ddof=1)
        y = (y - y.mean()) / y.std(ddof=1)
    else:
        x = (dd - dd.mean()) / dd.std(ddof=1)
        y = (cds - cds.mean()) / cds.std(ddof=1)

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': max(1, horizon - 1), 'use_correction': True}
    )
    coef  = model.params[1]        # slope = correlation (standardized)
    tstat = model.tvalues[1]
    pval  = model.pvalues[1]
    return coef, tstat, pval, n


# ──────────────────────────────────────────────────────────────
# COUNTRY-LEVEL RESULTS WITH NW TESTS
# ──────────────────────────────────────────────────────────────

def country_nw_results(df, horizon, spearman=False):
    """Returns DataFrame with one row per country: corr, t, p, n."""
    rows = []
    for country, grp in df.groupby(COUNTRY_COL):
        dd  = grp['delta_dd'].values
        cds = grp['delta_cds'].values
        coef, tstat, pval, n = nw_corr_test(dd, cds, horizon, spearman=spearman)
        rows.append({
            COUNTRY_COL: country,
            'corr': coef,
            'tstat': tstat,
            'pval': pval,
            'n': n
        })
    return pd.DataFrame(rows).set_index(COUNTRY_COL)


# ──────────────────────────────────────────────────────────────
# AGGREGATE TABLE WITH SE BANDS
# Group mean + bootstrap SE across countries (accounts for
# cross-sectional sampling uncertainty in the group average)
# ──────────────────────────────────────────────────────────────

def bootstrap_mean_ci(corrs, n_boot=2000, ci=0.95):
    """Bootstrap CI for the mean of a small group of correlations."""
    corrs = corrs.dropna().values
    if len(corrs) < 2:
        return np.nan, np.nan
    boot = [np.mean(np.random.choice(corrs, size=len(corrs), replace=True))
            for _ in range(n_boot)]
    lo = np.percentile(boot, 100*(1-ci)/2)
    hi = np.percentile(boot, 100*(1+ci)/2)
    return lo, hi


def run_nw_table(spearman=False):
    """
    Builds the full results table with NW-adjusted correlations,
    country-level significance, and bootstrapped group CIs.
    """
    corr_label = 'Spearman' if spearman else 'Pearson'
    records = []

    for h in HORIZONS:
        for m in PATHS.keys():
            changes = compute_changes(dfs[m], h)
            if changes.empty:
                continue

            country_res = country_nw_results(changes, h, spearman=spearman)
            country_res['group'] = country_res.index.map(
                lambda c: 'exporter' if c in EXPORTERS
                          else ('control' if c in CONTROLS else 'other')
            )

            for grp_name, grp_countries in [('all', EXPORTERS + CONTROLS),
                                             ('exporters', EXPORTERS),
                                             ('controls', CONTROLS)]:
                subset = country_res[country_res.index.isin(grp_countries)]
                mean_corr = subset['corr'].mean()
                lo, hi = bootstrap_mean_ci(subset['corr'])
                n_sig = (subset['pval'] < 0.10).sum()  # country-level sig at 10%

                records.append({
                    'horizon_w': h,
                    'model': m,
                    'group': grp_name,
                    'corr': mean_corr,
                    'ci_lo': lo,
                    'ci_hi': hi,
                    'n_sig_10pct': n_sig,
                    'n_countries': len(subset),
                    'corr_type': corr_label,
                })

    return pd.DataFrame(records)


np.random.seed(42)
nw_pearson  = run_nw_table(spearman=False)
nw_spearman = run_nw_table(spearman=True)
nw_results  = pd.concat([nw_pearson, nw_spearman], ignore_index=True)


# ──────────────────────────────────────────────────────────────
# DISPLAY
# ──────────────────────────────────────────────────────────────

def display_nw_table(df, corr_name, group_col, title):
    subset = df[(df['corr_type'] == corr_name) & (df['group'] == group_col)]
    print(f"\n{'='*70}")
    print(f"  {title} | NW-adjusted (lag = horizon - 1) | 95% bootstrap CI")
    print(f"{'='*70}")
    print(f"{'Horizon':>10} {'Model':>6} {'Corr':>8} {'95% CI':>18} {'N sig (10%)':>12}")
    print(f"{'-'*70}")
    for _, row in subset.sort_values(['horizon_w','model']).iterrows():
        print(f"{row['horizon_w']:>10}w {row['model']:>6} "
              f"{row['corr']:>8.3f} "
              f"[{row['ci_lo']:>6.3f}, {row['ci_hi']:>6.3f}] "
              f"{int(row['n_sig_10pct']):>6}/{int(row['n_countries'])}")


display_nw_table(nw_results, 'Spearman', 'exporters', 'Spearman — Exporters')
display_nw_table(nw_results, 'Spearman', 'controls',  'Spearman — Controls')
display_nw_table(nw_results, 'Pearson',  'exporters', 'Pearson  — Exporters')
display_nw_table(nw_results, 'Pearson',  'controls',  'Pearson  — Controls')

MissingDataError: exog contains inf or nans

In [ ]:
# ──────────────────────────────────────────────────────────────
# PER-COUNTRY CORRELATION TABLES
# Rows = countries (exporters first, then controls)
# Columns = M0 / M1 / M2
# One table per (correlation type × horizon)
# ──────────────────────────────────────────────────────────────

def per_country_table(horizon, spearman=False, nw=False):
    """
    Returns a DataFrame with countries as rows and models as columns.
    Values are NW-adjusted (if nw=True) or simple correlations.
    Countries ordered: exporters first, then controls.
    Significance stars added to NW estimates (*** p<0.01, ** p<0.05, * p<0.10).
    """
    corr_func = spearman_corr if not nw else None
    country_order = [c for c in EXPORTERS if c in all_countries] + \
                    [c for c in CONTROLS  if c in all_countries]

    model_cols = {}
    for m in PATHS.keys():
        changes = compute_changes(dfs[m], horizon)
        if changes.empty:
            model_cols[m] = pd.Series(dtype=float)
            continue

        if nw:
            res = country_nw_results(changes, horizon, spearman=spearman)
            # Format: value + stars
            def fmt(row):
                if pd.isna(row['corr']):
                    return '—'
                stars = '***' if row['pval'] < 0.01 else ('**' if row['pval'] < 0.05 else ('*' if row['pval'] < 0.10 else ''))
                return f"{row['corr']:.3f}{stars}"
            model_cols[m] = res.apply(fmt, axis=1)
        else:
            corrs = country_correlations(changes, spearman_corr if spearman else pearson_corr)
            model_cols[m] = corrs.round(3)

    table = pd.DataFrame(model_cols)
    # Reindex to enforce country order, keeping only present countries
    present = [c for c in country_order if c in table.index]
    table = table.reindex(present)

    # Add group separator label
    table.index = pd.Index([
        c + (" [E]" if c in EXPORTERS else " [C]") for c in present
    ], name='Country')

    return table


def display_per_country_table(horizon, spearman=False, nw=False):
    corr_name = 'Spearman' if spearman else 'Pearson'
    nw_note   = ' (NW-adjusted, * p<0.10, ** p<0.05, *** p<0.01)' if nw else ''
    print(f"\n{'='*65}")
    print(f"  {corr_name} per country — {horizon}w horizon{nw_note}")
    print(f"  [E] = Oil Exporter  |  [C] = Control")
    print(f"{'='*65}")
    tbl = per_country_table(horizon, spearman=spearman, nw=nw)
    print(tbl.to_string())
    print()
    return tbl


# ── Simple correlations (all horizons, both metrics) ──────────
print("\n" + "▓"*65)
print("  SIMPLE CORRELATIONS — PER COUNTRY")
print("▓"*65)
for h in HORIZONS:
    display_per_country_table(h, spearman=False, nw=False)
for h in HORIZONS:
    display_per_country_table(h, spearman=True,  nw=False)

# ── NW-adjusted with significance stars ───────────────────────
print("\n" + "▓"*65)
print("  NW-ADJUSTED CORRELATIONS — PER COUNTRY")
print("▓"*65)
for h in HORIZONS:
    display_per_country_table(h, spearman=False, nw=True)
for h in HORIZONS:
    display_per_country_table(h, spearman=True,  nw=True)



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  SIMPLE CORRELATIONS — PER COUNTRY
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Pearson per country — 1w horizon
  [E] = Oil Exporter  |  [C] = Control
                     M0     M1     M2
Country                              
Saudi Arabia [E]  0.003 -0.092 -0.096
Abu Dhabi [E]     0.008 -0.127 -0.180
Qatar [E]         0.004 -0.136 -0.164
Colombia [E]     -0.120 -0.203 -0.169
Mexico [E]       -0.281 -0.306 -0.278
Brazil [E]       -0.209 -0.281 -0.224
Egypt [E]         0.003 -0.013 -0.033
Malaysia [E]     -0.023 -0.199 -0.205
Chile [C]        -0.059 -0.205 -0.139
China [C]         0.005 -0.078  0.019
Indonesia [C]    -0.011 -0.115 -0.183
Philippines [C]  -0.000 -0.104 -0.198
South Africa [C] -0.251 -0.238 -0.205
South Korea [C]  -0.018 -0.069 -0.157
Thailand [C]      0.004 -0.152 -0.173
Turkey [C]       -0.110 -0.185 -0.106


  Pearson per country — 2w horizon
  [E] = Oil Exporter  |  [C] = Cont